# RQ2-v3 cross-batch variance validation

A separate CPU-only diagnostic using the two saved 8-batch shards from the T4×2 interaction probe. Fold A fits the Gradient-Oracle marginals on batches 0–7 and evaluates full q-aware importance-corrected variance on batches 8–15; Fold B swaps them. Geometry and Resource policies remain frozen. No model, checkpoint, GPU, accuracy, or test data is used.

In [ ]:
import os, subprocess, sys, json, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('CPU-only notebook; accelerator should be None')

## Locate the previously completed T4×2 interaction output

In [ ]:
import importlib, rq2_gradient_variance_v3, rq2_cross_batch_variance
rq2_gradient_variance_v3 = importlib.reload(rq2_gradient_variance_v3)
rq2_cross_batch_variance = importlib.reload(rq2_cross_batch_variance)
INTERACTION_ROOT = rq2_gradient_variance_v3.find_interaction_probe_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-rq2-cross-batch'
)
worker_metadata = sorted((INTERACTION_ROOT/'worker_shards').glob('*/metadata.json'))
assert len(worker_metadata) == 2, f'Expected two saved GPU shards, found: {worker_metadata}'
print('Interaction root:', INTERACTION_ROOT)
for path in worker_metadata:
    item = json.loads(path.read_text())
    print(path.parent.name, item['global_batch_indices'])

## Fit on 8 batches, evaluate on the other 8, then swap

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-cross-batch-variance')
summary = rq2_cross_batch_variance.run_cross_batch_variance_validation(
    INTERACTION_ROOT, OUTPUT_DIR
)
summary['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'cross_batch_variance_summary.json').write_text(json.dumps(summary, indent=2)+'\n')
print(json.dumps(summary, indent=2))

## Inspect held-out ordering and cross-fitted oracle

In [ ]:
import pandas as pd
from IPython.display import display, Image
by_fold = pd.read_csv(OUTPUT_DIR/'cross_batch_variance_by_fold.csv')
aggregate = pd.read_csv(OUTPUT_DIR/'cross_batch_variance_summary.csv')
oracle = pd.read_csv(OUTPUT_DIR/'cross_batch_oracle_marginals.csv')
display(by_fold)
display(aggregate)
display(oracle.pivot(index='width', columns='fold', values='crossfit_oracle_pi'))
display(Image(filename=str(OUTPUT_DIR/'cross_batch_variance.png')))

## Validate and export

In [ ]:
required = [
    'cross_batch_variance_by_fold.csv',
    'cross_batch_variance_summary.csv',
    'cross_batch_oracle_marginals.csv',
    'cross_batch_variance.png',
    'cross_batch_variance_summary.json',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing cross-batch artifacts: {missing}'
saved = json.loads((OUTPUT_DIR/'cross_batch_variance_summary.json').read_text())
assert saved['gradient_oracle_cross_fitted'] is True
assert saved['full_q_aware_variance_used'] is True
assert saved['training_performed'] is False and saved['gpu_used'] is False
bundle_path = Path('/kaggle/working/rq2-cross-batch-variance.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path